# 12 - CTD disjoint-split robustness runner

This experiment tests whether the distractor-robustness effect from Notebook 11 generalizes beyond chemical-disjoint evaluation.

Splits:
- ChemicalID-disjoint
- GeneID-disjoint
- DiseaseID-disjoint

For each split, compare Vanilla SFT vs Distractor-aware SFT using the same pilot evaluation: clean, distractor-5, and no-path-5.

Default configuration: 1 seed, 1,800 training examples, 80 optimization steps, and 100 evaluation examples per condition. Increase `SEEDS` or `RUN_ALL_SPLITS` for the paper-quality run.

In [ ]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"


In [ ]:
import os, random, re
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
CHEM_GENE='/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE='/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    print('Upload CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz')
    files.upload()
assert os.path.exists(CHEM_GENE) and os.path.exists(GENE_DISEASE)
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
chem_cols=['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols=['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']
chem=pd.read_csv(CHEM_GENE,sep='\t',comment='#',header=None,names=chem_cols,dtype=str,low_memory=False)
gd=pd.read_csv(GENE_DISEASE,sep='\t',comment='#',header=None,names=gd_cols,dtype=str,low_memory=False)
chem=chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].dropna(subset=['ChemicalName','ChemicalID','GeneSymbol','GeneID']).copy()
gd=gd.dropna(subset=['GeneID','DiseaseName','DiseaseID']).drop_duplicates(['GeneID','DiseaseID']).copy()
chem['GeneID']=chem['GeneID'].str.replace(r'\.0$','',regex=True)
gd['GeneID']=gd['GeneID'].str.replace(r'\.0$','',regex=True)
pairs=chem.merge(gd[['GeneID','DiseaseName','DiseaseID']],on='GeneID',how='inner')
pairs=pairs[['ChemicalName','ChemicalID','GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna().drop_duplicates(['ChemicalID','GeneID','DiseaseID']).reset_index(drop=True)
print('2-hop paths:',len(pairs))


In [ ]:
# -------------------- CONFIG --------------------
MODEL_NAME='Qwen/Qwen2.5-0.5B-Instruct'
SEEDS=[1]
RUN_ALL_SPLITS=True
MAX_TRAIN=1800
MAX_EVAL=100
MAX_STEPS=80
DROPOUT_PROB=0.5
# -----------------------------------------------

from transformers import AutoModelForCausalLM,AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def render(prompt,answer=None):
    msgs=[{'role':'user','content':prompt}]
    if answer is not None: msgs.append({'role':'assistant','content':answer})
    return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=answer is None)


In [ ]:
def make_split(df, split_key, seed):
    rng=random.Random(seed)
    ids=df[split_key].drop_duplicates().tolist()
    rng.shuffle(ids)
    n_eval=max(1,int(0.1*len(ids)))
    eval_ids=set(ids[:n_eval])
    train=df[~df[split_key].isin(eval_ids)].copy()
    eval_df=df[df[split_key].isin(eval_ids)].copy()
    return train, eval_df

def clean_prompt(row):
    return (f'Evidence 1: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n'
            f'Evidence 2: gene {row.GeneSymbol} is linked to disease {row.DiseaseName}.\n'
            f'Question: What disease is connected to {row.ChemicalName} through gene {row.GeneSymbol}? '
            'Return Disease: <name> and Path: Chemical -> Gene -> Disease.')

def clean_answer(row):
    return f'Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'

def build_edge_pool(df):
    return list({(str(g),str(d)) for g,d in df[['GeneSymbol','DiseaseName']].itertuples(index=False,name=None)})

def distractor_prompt(row,k,edge_pool,rng):
    candidates=[x for x in edge_pool if x[0]!=row.GeneSymbol and x[1]!=row.DiseaseName]
    if len(candidates)<k: return None
    ds=rng.sample(candidates,k)
    edges=[f'{row.GeneSymbol} -> {row.DiseaseName}']+[f'{g} -> {d}' for g,d in ds]
    rng.shuffle(edges)
    return (f'Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n'
            +'Gene-disease evidence:\n- '+'\n- '.join(edges)
            +f'\nQuestion: Using only the evidence above, what disease is connected to {row.ChemicalName} through gene {row.GeneSymbol}? '
             'Return Disease: <name> and Path: Chemical -> Gene -> Disease.')

def no_path_prompt(row,k,edge_pool,rng):
    candidates=[x for x in edge_pool if x[0]!=row.GeneSymbol and x[1]!=row.DiseaseName]
    if len(candidates)<k: return None
    ds=rng.sample(candidates,k)
    edges=[f'{g} -> {d}' for g,d in ds]; rng.shuffle(edges)
    return (f'Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n'
            +'Gene-disease evidence:\n- '+'\n- '.join(edges)
            +f'\nQuestion: Is there a supported disease path from {row.ChemicalName} through gene {row.GeneSymbol}? '
             'Answer YES or NO. If no supported path exists, say No supported path.')


In [ ]:
def make_train_dataset(train_df, edge_pool, condition, seed):
    rng=random.Random(seed)
    sample=train_df.sample(min(MAX_TRAIN,len(train_df)),random_state=seed).reset_index(drop=True)
    rows=[]
    for row in sample.itertuples(index=False):
        if condition=='vanilla' or rng.random()>DROPOUT_PROB:
            p,a=clean_prompt(row),clean_answer(row)
        else:
            p=distractor_prompt(row,5,edge_pool,rng)
            if p is None: p,a=clean_prompt(row),clean_answer(row)
            else: a=clean_answer(row)
        rows.append({'text':render(p,a)})
    return Dataset.from_list(rows)

def make_eval_sets(eval_df, edge_pool, seed):
    rng=random.Random(seed+1000)
    eval_df=eval_df.sample(min(MAX_EVAL,len(eval_df)),random_state=seed+1).reset_index(drop=True)
    sets={'clean':[],'distractor_5':[],'no_path_5':[]}
    for row in eval_df.itertuples(index=False):
        meta={'target_disease':row.DiseaseName,'target_gene':row.GeneSymbol,'target_chemical':row.ChemicalName}
        sets['clean'].append({**meta,'prompt':clean_prompt(row),'kind':'positive'})
        p=distractor_prompt(row,5,edge_pool,rng)
        if p: sets['distractor_5'].append({**meta,'prompt':p,'kind':'positive'})
        p=no_path_prompt(row,5,edge_pool,rng)
        if p: sets['no_path_5'].append({**meta,'prompt':p,'kind':'no_path'})
    return sets


In [ ]:
from peft import LoraConfig,PeftModel
from trl import SFTConfig,SFTTrainer
lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,target_modules=['q_proj','k_proj','v_proj','o_proj'],bias='none',task_type='CAUSAL_LM')

def train_adapter(ds,outdir):
    m=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda(); m.config.use_cache=False
    args=SFTConfig(output_dir=outdir,per_device_train_batch_size=4,gradient_accumulation_steps=2,max_steps=MAX_STEPS,learning_rate=2e-4,logging_steps=20,save_strategy='no',report_to='none',packing=False,gradient_checkpointing=False,fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    t=SFTTrainer(model=m,args=args,train_dataset=ds,processing_class=tokenizer,peft_config=lora); t.train(); t.save_model(outdir); tokenizer.save_pretrained(outdir)
    del t,m; torch.cuda.empty_cache()

def generate_batched(m,prompts,batch_size=16,max_new_tokens=56):
    m.eval(); outs=[]
    for s in range(0,len(prompts),batch_size):
        enc=tokenizer([render(p) for p in prompts[s:s+batch_size]],return_tensors='pt',padding=True,truncation=True,max_length=384)
        enc={k:v.to(m.device) for k,v in enc.items()}
        with torch.inference_mode(): out=m.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        n=enc['input_ids'].shape[1]; outs.extend(tokenizer.batch_decode(out[:,n:],skip_special_tokens=True))
    return outs

def score(items,preds):
    hits=[]
    for item,p in zip(items,preds):
        q=re.sub(r'[^a-z0-9]+',' ',p.lower())
        if item['kind']=='no_path': hits.append('no supported path' in q)
        else: hits.append(re.sub(r'[^a-z0-9]+',' ',item['target_disease'].lower()) in q)
    return sum(hits)/len(hits) if hits else float('nan')


In [ ]:
def run_one_split(split_name,split_key,seed):
    train_df,eval_df=make_split(pairs,split_key,seed)
    edge_pool=build_edge_pool(train_df)
    print('\nSPLIT',split_name,'seed',seed,'train',len(train_df),'eval',len(eval_df),'unique test',eval_df[split_key].nunique())
    results=[]
    for condition in ['vanilla','robust']:
        ds=make_train_dataset(train_df,edge_pool,condition,seed)
        outdir=f'./outputs/12-{split_name}-seed{seed}-{condition}'
        print('Training',condition)
        train_adapter(ds,outdir)
        base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda()
        m=PeftModel.from_pretrained(base,outdir); m.eval()
        eval_sets=make_eval_sets(eval_df,edge_pool,seed)
        for metric,items in eval_sets.items():
            preds=generate_batched(m,[x['prompt'] for x in items])
            score_val=score(items,preds)
            results.append({'split':split_name,'seed':seed,'condition':condition,'metric':metric,'score':score_val,'n_eval':len(items)})
            print(condition,metric,round(score_val,3),len(items))
        del m,base; torch.cuda.empty_cache()
    return results


In [ ]:
SPLITS={
    'ChemicalID':'ChemicalID',
    'GeneID':'GeneID',
    'DiseaseID':'DiseaseID',
}
if not RUN_ALL_SPLITS:
    SPLITS={'ChemicalID':'ChemicalID'}

all_results=[]
for split_name,split_key in SPLITS.items():
    for seed in SEEDS:
        all_results.extend(run_one_split(split_name,split_key,seed))

results_df=pd.DataFrame(all_results)
results_df.to_csv('12_results.csv',index=False)
summary=(results_df.groupby(['split','condition','metric'])['score'].agg(['mean','std','count','min','max']).reset_index())
summary.to_csv('12_summary.csv',index=False)
print('\nSUMMARY')
print(summary.to_string(index=False))


In [ ]:
print('\nROBUSTNESS DELTA')
print('='*88)
for split in results_df['split'].unique():
    sub=summary[summary['split']==split].pivot_table(index='metric',columns='condition',values='mean')
    if 'vanilla' in sub.columns and 'robust' in sub.columns:
        sub['delta']=sub['robust']-sub['vanilla']
    print('\n',split)
    print(sub)


## Interpretation

The key paper-quality question is whether distractor-aware training continues to help when the held-out split is made stricter. In particular, GeneID-disjoint asks whether the effect survives when the intermediate gene entities are unseen during training; DiseaseID-disjoint asks whether the effect survives when the outcome entities are unseen.

This pilot defaults to one seed for speed. Before drawing final conclusions, repeat with at least 3 seeds and report mean +/- standard deviation or confidence intervals.